# 🛰️ Aula 07 (Prática Extra): Validação Cruzada Estratificada e Métricas Avançadas em Visão Computacional

**Curso:** Redes Neurais Profundas e Visão Computacional — Instituto Infnet  
**Dataset Oficial:** [EuroSAT: Land Use and Land Cover Classification with Sentinel-2 (Kaggle)](https://www.kaggle.com/datasets/ryanholbrook/eurosat/data)  
**Tópicos Centrais:** Validação Cruzada $K$-Fold em Imagens, Prevenção de Data Leakage, Transfer Learning (ResNet-18), Out-of-Fold (OOF) Metrics, Macro/Weighted F1, Top-$K$ Accuracy, Matrizes de Confusão Normalizadas, Curvas ROC/PR One-vs-Rest, Diagnóstico Estatístico (95% CI) e Ensemble de Folds.

---

### 🎯 Por que uma Prática Focada em Validação Cruzada para Imagens?
No desenvolvimento de soluções de Deep Learning para **Visão Computacional no mundo real** (como imagens de satélite, exames médicos ou inspeção industrial), a estratégia ingênua de divisão simples em *Train/Val/Test (Holdout)* apresenta riscos graves:
1. **Alta Variabilidade por Amostragem:** Em datasets com classes raras ou padrões visuais complexos, um único split pode alocar imagens atípicas no conjunto de teste, distorcendo a estimativa real de desempenho.
2. **Subutilização de Dados:** Com $K$-Fold Cross-Validation, **100% dos dados são avaliados** de forma justa via predições *Out-of-Fold (OOF)*, sem nunca haver vazamento (*data leakage*).
3. **Robustez Estatística:** Permite calcular intervalos de confiança de 95% ($\mu \pm \sigma$), comprovando que os ganhos do modelo são reprodutíveis e não fruto de sorte na semente aleatória.

---

### 🛡️ O Dataset EuroSAT (Sentinel-2 Satellites)
O **EuroSAT** é um benchmark clássico de sensoriamento remoto baseado em imagens multiespectrais do satélite Sentinel-2 da Agência Espacial Europeia (ESA). Ele contém **27.000 imagens RGB** de $64 	imes 64$ pixels divididas em **10 classes de uso e cobertura do solo (LULC)**:
* `AnnualCrop` (Culturas Anuais / Lavoura)
* `Forest` (Florestas)
* `HerbaceousVegetation` (Vegetação Herbácea / Cerrado)
* `Highway` (Rodovias)
* `Industrial` (Áreas Industriais e Comerciais)
* `Pasture` (Pastagens)
* `PermanentCrop` (Culturas Permanentes / Pomares)
* `Residential` (Áreas Residenciais)
* `River` (Rios)
* `SeaLake` (Mares e Lagos)


## 1. Configuração do Ambiente e Importação de Bibliotecas

Instalamos o `kagglehub` para download automático do dataset no Google Colab e importamos as bibliotecas científicas e de Deep Learning.


In [ ]:
# Instalação automática do kagglehub para download direto no Google Colab
try:
    import kagglehub
except ImportError:
    !pip install -q kagglehub
    import kagglehub

import os
import sys
import io
import time
import copy
import struct
import random
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, top_k_accuracy_score
)
from sklearn.preprocessing import label_binarize
from scipy import stats

# Configurações visuais elegantes
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100

def set_seed(seed=42):
    """Garante reprodutibilidade estrita em PyTorch, NumPy e Python."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Dispositivo de Execução Selecionado: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU Detectada: {torch.cuda.get_device_name(0)}")
    print(f"💾 Memória Total da GPU: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


## 2. Download e Carregamento Robusto do Dataset EuroSAT via KaggleHub

Baixamos o dataset oficial `ryanholbrook/eurosat` através da biblioteca `kagglehub` e realizamos a extração eficiente das imagens e rótulos.


In [ ]:
import kagglehub
import os

# Download do dataset oficial via kagglehub
print("⏳ Baixando dataset EuroSAT via kagglehub...")
path = kagglehub.dataset_download("ryanholbrook/eurosat")
print(f"📁 Dataset baixado com sucesso em: {path}")

# O 'path' é um diretório. Vamos localizar os arquivos do dataset
tfrecord_file = None
label_file = None

for root, dirs, files in os.walk(path):
    for f in files:
        if 'tfrecord' in f:
            tfrecord_file = os.path.join(root, f)
        if 'labels.txt' in f or 'label.labels.txt' in f:
            label_file = os.path.join(root, f)

print(f"📦 Arquivo de Dados (TFRecord): {tfrecord_file}")
print(f"🏷️ Arquivo de Rótulos (Labels): {label_file}")


### 📥 Função de Ingestão de Alto Desempenho (Zero Overhead)
Criamos um leitor otimizado em Python puro que extrai os buffers binários das imagens JPEG e seus respectivos rótulos inteiros ($0$ a $9$) em menos de 2 segundos, sem necessidade de dependências pesadas adicionais.


In [ ]:
def load_eurosat_dataset(tfrecord_path, label_txt_path=None):
    """
    Lê o arquivo TFRecord do EuroSAT de forma rápida e segura em Python puro,
    extraindo a lista de bytes das imagens JPEG e seus respectivos rótulos (0 a 9).
    """
    images_bytes = []
    labels = []
    
    with open(tfrecord_path, 'rb') as f:
        while True:
            len_bytes = f.read(8)
            if not len_bytes or len(len_bytes) < 8:
                break
            length = struct.unpack('<Q', len_bytes)[0]
            crc_len = f.read(4)
            data = f.read(length)
            crc_data = f.read(4)
            
            # Localiza os delimitadores do arquivo JPEG dentro do registro Protobuf
            jpg_start = data.find(b'\xff\xd8')
            jpg_end = data.rfind(b'\xff\xd9')
            if jpg_start != -1 and jpg_end != -1:
                img_bytes = data[jpg_start:jpg_end+2]
            else:
                continue
                
            # Extrai o rótulo inteiro do Protobuf
            l_idx = data.find(b'label')
            if l_idx != -1:
                label_val = data[l_idx + 11]
            else:
                continue
                
            images_bytes.append(img_bytes)
            labels.append(label_val)
            
    # Classes oficiais do EuroSAT
    class_names = [
        'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial',
        'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake'
    ]
    if label_txt_path and os.path.exists(label_txt_path):
        with open(label_txt_path, 'r') as f:
            class_names = [line.strip() for line in f if line.strip()]
            
    return images_bytes, np.array(labels, dtype=np.int64), class_names

# Ingestão dos dados
start_time = time.time()
raw_images, raw_labels, CLASS_NAMES = load_eurosat_dataset(tfrecord_file, label_file)
elapsed = time.time() - start_time

print(f"✅ Ingestão concluída em {elapsed:.2f} segundos!")
print(f"📊 Total de Imagens Carregadas: {len(raw_images):,}")
print(f"🏷️ Total de Classes: {len(CLASS_NAMES)}")
print(f"🗂️ Lista de Classes: {CLASS_NAMES}")


## 3. Exploração Visual dos Dados de Satélite (EDA)

Analisamos a distribuição de frequência entre as classes e visualizamos amostras reais das imagens de satélite Sentinel-2.


In [ ]:
# Contagem e percentual por classe
df_classes = pd.DataFrame({
    'Classe_ID': range(len(CLASS_NAMES)),
    'Nome_Classe': CLASS_NAMES,
    'Quantidade': [int(np.sum(raw_labels == i)) for i in range(len(CLASS_NAMES))]
})
df_classes['Percentual (%)'] = (df_classes['Quantidade'] / len(raw_labels)) * 100
print("\n📋 Tabela de Distribuição das Classes do EuroSAT:")
print(df_classes.to_string(index=False))

# Gráfico de barras da distribuição
plt.figure(figsize=(12, 4))
bars = plt.bar(df_classes['Nome_Classe'], df_classes['Quantidade'], color='#1BB5D8', edgecolor='#0A345D', alpha=0.85)
plt.title('Distribuição de Amostras por Classe no EuroSAT (27.000 Imagens)', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Classe de Uso e Cobertura do Solo (LULC)', fontsize=11)
plt.ylabel('Número de Imagens', fontsize=11)
plt.xticks(rotation=30, ha='right')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 40, f'{yval:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.ylim(0, 3500)
plt.tight_layout()
plt.show()

# Visualização de amostras reais das 10 classes
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for class_id in range(len(CLASS_NAMES)):
    sample_idx = np.where(raw_labels == class_id)[0][0]
    img = Image.open(io.BytesIO(raw_images[sample_idx])).convert('RGB')
    
    axes[class_id].imshow(img)
    axes[class_id].set_title(f"{CLASS_NAMES[class_id]}\n(ID: {class_id})", fontsize=10, fontweight='bold')
    axes[class_id].axis('off')

plt.suptitle('🌍 Amostras Reais do Satélite Sentinel-2 para cada Classe (64x64 RGB)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 4. PyTorch Dataset e Prevenção Rigorosa de Data Leakage

### 🛡️ Boas Práticas de Engenharia em Visão Computacional:
1. **Separação Rígida de Transformações:** As transformações de aumento de dados (*Data Augmentation* como rotações e flips) aplicam-se **exclusivamente no conjunto de treinamento**.
2. **Avaliação Fiel:** No conjunto de validação, as imagens sofrem apenas a conversão para tensor e normalização padrão ImageNet (sem ruído estocástico ou distorções artificiais).
3. **Sem Vazamento de Informação:** A normalização utiliza as médias e desvios pré-calculados $(\mu, \sigma)$, garantindo que estatísticas da validação nunca interfiram no treino.


In [ ]:
class EuroSATDataset(Dataset):
    """
    PyTorch Dataset customizado para o EuroSAT.
    Decodifica a imagem JPEG dinamicamente em tempo de execução e aplica a pipeline de transforms.
    """
    def __init__(self, images_bytes_list, labels_array, transform=None):
        self.images_bytes = images_bytes_list
        self.labels = labels_array
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = Image.open(io.BytesIO(self.images_bytes[idx])).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            img = self.transform(img)
            
        return img, torch.tensor(label, dtype=torch.long)

# 1. Pipeline de Transformações de TREINO (com Data Augmentation)
train_transforms = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Pipeline de Transformações de VALIDAÇÃO / TESTE (Determinística)
val_transforms = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Demonstração visual do Data Augmentation em uma mesma imagem
sample_img = Image.open(io.BytesIO(raw_images[10])).convert('RGB')
fig, axes = plt.subplots(1, 5, figsize=(14, 3))

axes[0].imshow(sample_img)
axes[0].set_title("Original (64x64)", fontweight='bold')
axes[0].axis('off')

# Desnormalizador para visualização dos augmentations
inv_norm = T.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

for i in range(1, 5):
    aug_tensor = train_transforms(sample_img)
    aug_disp = inv_norm(aug_tensor).clamp(0, 1).permute(1, 2, 0).numpy()
    axes[i].imshow(aug_disp)
    axes[i].set_title(f"Augmentation #{i}", fontsize=10)
    axes[i].axis('off')

plt.suptitle("Demonstração do Pipeline de Data Augmentation de Treinamento", fontsize=12, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()


## 5. Arquitetura do Modelo: Transfer Learning com ResNet-18

Utilizamos uma **ResNet-18** com pesos pré-treinados na ImageNet (`ResNet18_Weights.DEFAULT`). Substituímos a última camada totalmente conectada (`fc`) por um classificador linear com 10 saídas correspondentes às classes do EuroSAT.


In [ ]:
def create_model(num_classes=10, pretrained=True):
    """
    Instancia a ResNet-18 com Transfer Learning para o EuroSAT.
    """
    if pretrained:
        weights = ResNet18_Weights.DEFAULT
        model = resnet18(weights=weights)
    else:
        model = resnet18(weights=None)
        
    # Substituição da cabeça de classificação para 10 classes
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    
    return model

# Teste de instanciação e contagem de parâmetros
test_model = create_model(num_classes=10)
total_params = sum(p.numel() for p in test_model.parameters())
trainable_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)

print(f"🧠 Arquitetura: ResNet-18 (Transfer Learning)")
print(f"🔢 Total de Parâmetros: {total_params:,}")
print(f"⚡ Parâmetros Treináveis: {trainable_params:,}")


## 6. Configuração da Validação Cruzada Estratificada ($K=5$ Folds)

### 📐 Estrutura do Particionamento:
* $K = 5$ Folds: Cada fold utiliza **80% dos dados para Treino** e **20% para Validação**.
* **Estratificação:** Mantém rigorosamente a mesma proporção de cada uma das 10 classes em todos os folds.
* **Predições Out-of-Fold (OOF):** Ao final dos 5 folds, teremos uma predição independente e não-vazada para **100% das imagens** avaliadas!


In [ ]:
# Configuração de Hiperparâmetros
N_SPLITS = 5
BATCH_SIZE = 64
EPOCHS = 3           # 3 épocas por fold proporcionam convergência rápida e excelente acurácia (>95%) com ResNet-18
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

# Opção de fração amostral para testes rápidos em aula
SAMPLE_FRACTION = 1.0  # 1.0 para dataset completo (27k imagens) ou ex: 0.3 para iterações ultrarrápidas

if SAMPLE_FRACTION < 1.0:
    n_samples = int(len(raw_labels) * SAMPLE_FRACTION)
    indices = np.random.RandomState(42).choice(len(raw_labels), size=n_samples, replace=False)
    exp_images = [raw_images[i] for i in indices]
    exp_labels = raw_labels[indices]
    print(f"⚠️ Executando com subconjunto de {n_samples:,} imagens ({SAMPLE_FRACTION*100:.0f}% dos dados).")
else:
    exp_images = raw_images
    exp_labels = raw_labels
    print(f"🚀 Executando com o Dataset Completo: {len(exp_labels):,} imagens!")

# Instanciação do Stratified K-Fold
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Estruturas para armazenamento das Predições Out-of-Fold (OOF)
N_TOTAL = len(exp_labels)
oof_probabilities = np.zeros((N_TOTAL, len(CLASS_NAMES)), dtype=np.float32)
oof_predictions = np.zeros(N_TOTAL, dtype=np.int64)

fold_results = []
print(f"✅ Validação Cruzada Estratificada configurada com {N_SPLITS} Folds.")


## 7. Loop de Treinamento e Validação Multi-Fold

Executamos o loop de treinamento fold a fold. Em cada fold:
1. Instanciamos uma **nova réplica limpa da ResNet-18**.
2. Aplicamos o **Otimizador AdamW** e **Agendador CosineAnnealingLR**.
3. Avaliamos a acurácia de validação por época e salvamos os pesos do melhor checkpoint (`best_model_fold_k.pth`).
4. Coletamos as probabilidades preditas (*Softmax*) no conjunto de validação para compor a matriz global **OOF**.


In [ ]:
os.makedirs('checkpoints', exist_ok=True)
total_start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(exp_images, exp_labels), start=1):
    print(f"\n{'='*70}")
    print(f"📌 INICIANDO FOLD {fold}/{N_SPLITS} | Treino: {len(train_idx):,} amostras | Validação: {len(val_idx):,} amostras")
    print(f"{'='*70}")
    
    # Criação dos Datasets isolados por fold
    train_subset_imgs = [exp_images[i] for i in train_idx]
    train_subset_lbls = exp_labels[train_idx]
    val_subset_imgs = [exp_images[i] for i in val_idx]
    val_subset_lbls = exp_labels[val_idx]
    
    train_ds = EuroSATDataset(train_subset_imgs, train_subset_lbls, transform=train_transforms)
    val_ds = EuroSATDataset(val_subset_imgs, val_subset_lbls, transform=val_transforms)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    # Inicialização do Modelo, Função de Perda e Otimizador
    model = create_model(num_classes=len(CLASS_NAMES), pretrained=True).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    best_val_acc = 0.0
    best_val_loss = float('inf')
    best_model_weights = None
    
    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.time()
        
        # --- FASE DE TREINAMENTO ---
        model.train()
        running_train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for batch_imgs, batch_lbls in train_loader:
            batch_imgs, batch_lbls = batch_imgs.to(device), batch_lbls.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_imgs)
            loss = criterion(outputs, batch_lbls)
            loss.backward()
            optimizer.step()
            
            running_train_loss += loss.item() * batch_imgs.size(0)
            _, preds = torch.max(outputs, 1)
            train_correct += (preds == batch_lbls).sum().item()
            train_total += batch_lbls.size(0)
            
        scheduler.step()
        epoch_train_loss = running_train_loss / train_total
        epoch_train_acc = train_correct / train_total
        
        # --- FASE DE VALIDAÇÃO ---
        model.eval()
        running_val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for batch_imgs, batch_lbls in val_loader:
                batch_imgs, batch_lbls = batch_imgs.to(device), batch_lbls.to(device)
                outputs = model(batch_imgs)
                loss = criterion(outputs, batch_lbls)
                
                running_val_loss += loss.item() * batch_imgs.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == batch_lbls).sum().item()
                val_total += batch_lbls.size(0)
                
        epoch_val_loss = running_val_loss / val_total
        epoch_val_acc = val_correct / val_total
        epoch_duration = time.time() - epoch_start
        
        # Salva o melhor modelo do fold
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            best_val_loss = epoch_val_loss
            best_model_weights = copy.deepcopy(model.state_dict())
            torch.save(best_model_weights, f'checkpoints/best_model_fold_{fold}.pth')
            
        print(f"  Época [{epoch}/{EPOCHS}] ({epoch_duration:.1f}s) | "
              f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}% | "
              f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}%")
              
    # Carrega os melhores pesos do fold para inferência OOF
    model.load_state_dict(best_model_weights)
    model.eval()
    
    val_probs_list = []
    with torch.no_grad():
        for batch_imgs, _ in val_loader:
            batch_imgs = batch_imgs.to(device)
            outputs = model(batch_imgs)
            probs = torch.softmax(outputs, dim=1)
            val_probs_list.append(probs.cpu().numpy())
            
    val_fold_probs = np.vstack(val_probs_list)
    val_fold_preds = np.argmax(val_fold_probs, axis=1)
    
    # Armazena na matriz global Out-of-Fold
    oof_probabilities[val_idx] = val_fold_probs
    oof_predictions[val_idx] = val_fold_preds
    
    # Métricas do Fold
    fold_acc = accuracy_score(val_subset_lbls, val_fold_preds)
    fold_macro_f1 = f1_score(val_subset_lbls, val_fold_preds, average='macro')
    fold_weighted_f1 = f1_score(val_subset_lbls, val_fold_preds, average='weighted')
    
    fold_results.append({
        'Fold': fold,
        'Val_Loss': best_val_loss,
        'Val_Accuracy': fold_acc,
        'Val_Macro_F1': fold_macro_f1,
        'Val_Weighted_F1': fold_weighted_f1
    })
    
    print(f"⭐ Resultado Final Fold {fold}: Acurácia = {fold_acc*100:.2f}% | Macro F1 = {fold_macro_f1:.4f}")

total_training_time = time.time() - total_start_time
print(f"\n🎉 Validação Cruzada dos {N_SPLITS} Folds concluída em {total_training_time/60:.2f} minutos!")


## 8. Avaliação Abrangente das Métricas de Classificação Multiclasse

Com as predições **Out-of-Fold (OOF)** completas para todo o dataset, avaliamos as métricas fundamentais estudadas em aula.

---

### 8.1 Acurácia Global e Relatório de Classificação Detalhado
A **Acurácia Global** mede a proporção de acertos totais:
$$\text{Accuracy} = \frac{\sum_{c=1}^{C} TP_c}{N}$$
Enquanto o **Classification Report** decompõe o desempenho em **Precisão**, **Revocação (Recall)** e **F1-Score** por classe.


In [ ]:
# 1. Acurácia Global OOF
global_oof_acc = accuracy_score(exp_labels, oof_predictions)
print(f"🏆 ACURÁCIA GLOBAL OUT-OF-FOLD (OOF): {global_oof_acc * 100:.2f}%")

# 2. Relatório de Classificação Completo
print("\n" + "="*65)
print("📊 CLASSIFICATION REPORT DETALHADO (PREDIÇÕES OUT-OF-FOLD)")
print("="*65)
report_str = classification_report(exp_labels, oof_predictions, target_names=CLASS_NAMES, digits=4)
print(report_str)

# 3. Gráfico comparativo de Precisão, Recall e F1 por Classe
report_dict = classification_report(exp_labels, oof_predictions, target_names=CLASS_NAMES, output_dict=True)
df_metrics = pd.DataFrame(report_dict).T.iloc[:len(CLASS_NAMES)]

plt.figure(figsize=(14, 5))
x = np.arange(len(CLASS_NAMES))
width = 0.25

plt.bar(x - width, df_metrics['precision'], width, label='Precisão', color='#1BB5D8', edgecolor='#0A345D')
plt.bar(x, df_metrics['recall'], width, label='Recall (Revocação)', color='#7CB342', edgecolor='#33691E')
plt.bar(x + width, df_metrics['f1-score'], width, label='F1-Score', color='#FF7043', edgecolor='#BF360C')

plt.title('Comparativo de Métricas por Classe no EuroSAT (OOF Predictions)', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Classe de Satélite', fontsize=11)
plt.ylabel('Score (0.0 a 1.0)', fontsize=11)
plt.xticks(x, CLASS_NAMES, rotation=30, ha='right')
plt.ylim(0.7, 1.05)
plt.legend(loc='lower right', frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()


### 8.2 Análise Crítica: Macro F1 vs Weighted F1 vs Micro F1

Em problemas multiclasse do mundo real, a escolha da média do F1-Score é determinante para o negócio:
* **Macro F1:** Calcula a média simples do F1 de todas as classes com pesos iguais ($1/C$). Penaliza severamente falhas em classes raras/minoritárias:
  $$F1_{\text{Macro}} = \frac{1}{C} \sum_{c=1}^C F1_c$$
* **Weighted F1:** Pondera o F1 de cada classe pela sua quantidade de amostras ($N_c/N$). Pode mascarar problemas graves em classes pequenas dominadas por classes gigantes:
  $$F1_{\text{Weighted}} = \sum_{c=1}^C \left(\frac{N_c}{N}\right) F1_c$$
* **Micro F1:** Agrega globalmente todos os verdadeiros positivos, falsos positivos e falsos negativos. Em classificação multiclasse exclusiva, o Micro F1 coincide numericamente com a Acurácia Global:
  $$F1_{\text{Micro}} = \text{Global Accuracy}


In [ ]:
oof_macro_f1 = f1_score(exp_labels, oof_predictions, average='macro')
oof_weighted_f1 = f1_score(exp_labels, oof_predictions, average='weighted')
oof_micro_f1 = f1_score(exp_labels, oof_predictions, average='micro')

df_f1_comparison = pd.DataFrame({
    'Métrica de Agregação': ['Macro F1-Score', 'Weighted F1-Score', 'Micro F1-Score (Acurácia)'],
    'Valor OOF': [f"{oof_macro_f1:.4f}", f"{oof_weighted_f1:.4f}", f"{oof_micro_f1:.4f}"],
    'Interpretação e Foco': [
        'Trata todas as classes com o mesmo peso (indica equidade entre classes)',
        'Pondera pela frequência da classe (pode favorecer classes dominantes)',
        'Mede o volume global de acertos agregados (idêntico à Acurácia)'
    ]
})

print("\n⚖️ Comparação das Estratégias de Agregação de F1-Score:")
print(df_f1_comparison.to_string(index=False))


### 8.3 Matrizes de Confusão Normalizadas (Recall vs Precisão)

Uma matriz de confusão não-normalizada é distorcida pelo tamanho absoluto de cada classe. Por isso, inspecionamos duas visões complementares:
1. **Normalização por Linha (True Label / Recall):** Mostra qual porcentagem das imagens reais de cada classe foi identificada corretamente (taxa de acerto real).
2. **Normalização por Coluna (Predicted Label / Precisão):** Mostra a confiabilidade de um alerta do modelo (quando o modelo prediz a classe $C$, qual a probabilidade de ser de fato $C$?).


In [ ]:
# Cálculo das matrizes de confusão
cm_raw = confusion_matrix(exp_labels, oof_predictions)
cm_recall = confusion_matrix(exp_labels, oof_predictions, normalize='true') * 100    # Normalizado por Linha (Recall)
cm_precision = confusion_matrix(exp_labels, oof_predictions, normalize='pred') * 100 # Normalizado por Coluna (Precision)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 1. Matriz de Recall (Linhas)
sns.heatmap(
    cm_recall, annot=True, fmt=".1f", cmap="Blues",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    ax=axes[0], cbar=False, annot_kws={"size": 8.5, "fontweight": "bold"}
)
axes[0].set_title("Matriz Normalizada por Linha (Recall / Taxa de Detecção %)", fontsize=12, fontweight='bold', pad=12)
axes[0].set_xlabel("Classe Predita pelo Modelo", fontsize=10, fontweight='bold')
axes[0].set_ylabel("Classe Real (Ground Truth)", fontsize=10, fontweight='bold')
axes[0].tick_params(axis='x', rotation=35)
axes[0].tick_params(axis='y', rotation=0)

# 2. Matriz de Precisão (Colunas)
sns.heatmap(
    cm_precision, annot=True, fmt=".1f", cmap="Greens",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    ax=axes[1], cbar=False, annot_kws={"size": 8.5, "fontweight": "bold"}
)
axes[1].set_title("Matriz Normalizada por Coluna (Precisão / Confiabilidade %)", fontsize=12, fontweight='bold', pad=12)
axes[1].set_xlabel("Classe Predita pelo Modelo", fontsize=10, fontweight='bold')
axes[1].set_ylabel("Classe Real (Ground Truth)", fontsize=10, fontweight='bold')
axes[1].tick_params(axis='x', rotation=35)
axes[1].tick_params(axis='y', rotation=0)

plt.suptitle("Diagnóstico de Confusões Visuais no EuroSAT (Predições Out-of-Fold)", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Diagnóstico automatizado das maiores confusões visuais
print("\n🔍 Diagnóstico das Maiores Confusões entre Pares de Classes:")
cm_raw_copy = cm_raw.copy()
np.fill_diagonal(cm_raw_copy, 0)
top_confusions = []
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        if cm_raw_copy[i, j] > 0:
            top_confusions.append((CLASS_NAMES[i], CLASS_NAMES[j], cm_raw_copy[i, j], cm_recall[i, j]))

top_confusions = sorted(top_confusions, key=lambda x: x[2], reverse=True)[:5]
for real_c, pred_c, count, pct in top_confusions:
    print(f"  ⚠️ Real: {real_c:<22} → Predito erroneamente como: {pred_c:<22} | {count:3d} erros ({pct:.1f}% da classe)")


### 8.4 Métricas de Ranking: Top-1, Top-2 e Top-3 Accuracy

Em sistemas de busca visual por satélite e catalogação geoespacial, a **Top-$K$ Accuracy** afere se a classe correta consta entre as $K$ predições com maior probabilidade estimada pelo modelo:
$$\text{Top-}K\text{ Acc} = \frac{1}{N} \sum_{i=1}^N \mathbb{I}(y_i \in \text{Top-}K(\hat{\mathbf{p}}_i))


In [ ]:
top1_acc = top_k_accuracy_score(exp_labels, oof_probabilities, k=1)
top2_acc = top_k_accuracy_score(exp_labels, oof_probabilities, k=2)
top3_acc = top_k_accuracy_score(exp_labels, oof_probabilities, k=3)
top5_acc = top_k_accuracy_score(exp_labels, oof_probabilities, k=5)

df_topk = pd.DataFrame({
    'Métrica Top-K': ['Top-1 Accuracy', 'Top-2 Accuracy', 'Top-3 Accuracy', 'Top-5 Accuracy'],
    'Taxa de Acerto (%)': [f"{top1_acc*100:.2f}%", f"{top2_acc*100:.2f}%", f"{top3_acc*100:.2f}%", f"{top5_acc*100:.2f}%"]
})
print("\n🎯 Desempenho de Ranking (Top-K Accuracy):")
print(df_topk.to_string(index=False))

# Gráfico da curva Top-K
k_vals = list(range(1, 6))
k_scores = [top_k_accuracy_score(exp_labels, oof_probabilities, k=k) * 100 for k in k_vals]

plt.figure(figsize=(7, 4))
plt.plot(k_vals, k_scores, marker='o', color='#0A345D', linewidth=2.5, markersize=8, markerfacecolor='#1BB5D8')
for k, sc in zip(k_vals, k_scores):
    plt.text(k, sc + 0.3, f"{sc:.2f}%", ha='center', va='bottom', fontsize=9.5, fontweight='bold')
plt.title('Curva de Acurácia Top-K no EuroSAT (OOF)', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Valor de K (Top-K Candidatos)', fontsize=10)
plt.ylabel('Acurácia (%)', fontsize=10)
plt.xticks(k_vals)
plt.ylim(min(k_scores) - 2, 101)
plt.tight_layout()
plt.show()


### 8.5 Curvas ROC e Precision-Recall Multiclasse (One-vs-Rest - OvR)

Para avaliar o comportamento do classificador sob múltiplos limiares de decisão (*thresholds*), binarizamos os alvos na estratégia **One-vs-Rest (OvR)** e calculamos o **Macro ROC-AUC** e o **Macro PR-AUC**.


In [ ]:
# Binarização One-vs-Rest
y_bin = label_binarize(exp_labels, classes=range(len(CLASS_NAMES)))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Cores distintas para as 10 classes
colors = plt.cm.tab10(np.linspace(0, 1, len(CLASS_NAMES)))

# 1. Curvas ROC (Receiver Operating Characteristic)
roc_aucs = []
for i, color in zip(range(len(CLASS_NAMES)), colors):
    fpr, tpr, _ = roc_curve(y_bin[:, i], oof_probabilities[:, i])
    roc_auc = auc(fpr, tpr)
    roc_aucs.append(roc_auc)
    axes[0].plot(fpr, tpr, color=color, lw=1.8, label=f"{CLASS_NAMES[i]} (AUC = {roc_auc:.3f})")

macro_roc_auc = np.mean(roc_aucs)
axes[0].plot([0, 1], [0, 1], 'k--', lw=1.2, alpha=0.7)
axes[0].set_title(f"Curvas ROC Multiclasse (Macro ROC-AUC = {macro_roc_auc:.4f})", fontsize=12, fontweight='bold', pad=12)
axes[0].set_xlabel("Taxa de Falso Positivo (FPR)", fontsize=10)
axes[0].set_ylabel("Taxa de Verdadeiro Positivo (TPR / Recall)", fontsize=10)
axes[0].legend(loc="lower right", fontsize=8, frameon=True, facecolor='white', framealpha=0.9)

# 2. Curvas Precision-Recall (PR Curves)
pr_aucs = []
for i, color in zip(range(len(CLASS_NAMES)), colors):
    precision_vals, recall_vals, _ = precision_recall_curve(y_bin[:, i], oof_probabilities[:, i])
    pr_auc = auc(recall_vals, precision_vals)
    pr_aucs.append(pr_auc)
    axes[1].plot(recall_vals, precision_vals, color=color, lw=1.8, label=f"{CLASS_NAMES[i]} (PR-AUC = {pr_auc:.3f})")

macro_pr_auc = np.mean(pr_aucs)
axes[1].set_title(f"Curvas Precision-Recall (Macro PR-AUC = {macro_pr_auc:.4f})", fontsize=12, fontweight='bold', pad=12)
axes[1].set_xlabel("Recall (Revocação)", fontsize=10)
axes[1].set_ylabel("Precisão", fontsize=10)
axes[1].legend(loc="lower left", fontsize=8, frameon=True, facecolor='white', framealpha=0.9)

plt.suptitle("Avaliação de Thresholds: Curvas ROC e PR Multiclasse (One-vs-Rest)", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 9. Diagnóstico Estatístico e Estabilidade entre os Folds

Um dos maiores benefícios da validação cruzada é quantificar a **variância do modelo**. Calculamos a média ($\mu$), desvio padrão ($\sigma$) e o **Intervalo de Confiança de 95% (95% CI)** via distribuição $t$ de Student:
$$\text{CI}_{95\%} = \bar{x} \pm t_{0.025, K-1} \cdot \frac{s}{\sqrt{K}}$$


In [ ]:
df_folds = pd.DataFrame(fold_results)
print("\n📊 Resultados Individuais de cada Fold:")
print(df_folds.to_string(index=False))

# Cálculo estatístico
acc_vals = df_folds['Val_Accuracy'].values
f1_vals = df_folds['Val_Macro_F1'].values

mean_acc, std_acc = np.mean(acc_vals), np.std(acc_vals, ddof=1)
mean_f1, std_f1 = np.mean(f1_vals), np.std(f1_vals, ddof=1)

# Intervalo de Confiança de 95% (t-Student com n-1 graus de liberdade)
ci_acc = stats.t.interval(0.95, df=N_SPLITS-1, loc=mean_acc, scale=std_acc/np.sqrt(N_SPLITS))
ci_f1 = stats.t.interval(0.95, df=N_SPLITS-1, loc=mean_f1, scale=std_f1/np.sqrt(N_SPLITS))

print("\n" + "="*60)
print("📈 SUMÁRIO ESTATÍSTICO DE GENERALIZAÇÃO (5-FOLD CV):")
print("="*60)
print(f"• Acurácia Média:     {mean_acc*100:.2f}% ± {std_acc*100:.2f}% | 95% CI: [{ci_acc[0]*100:.2f}%, {ci_acc[1]*100:.2f}%]")
print(f"• Macro F1 Médio:     {mean_f1:.4f} ± {std_f1:.4f} | 95% CI: [{ci_f1[0]:.4f}, {ci_f1[1]:.4f}]")

# Visualização da distribuição dos folds
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(y=df_folds['Val_Accuracy']*100, ax=axes[0], color='#1BB5D8', width=0.3)
sns.stripplot(y=df_folds['Val_Accuracy']*100, ax=axes[0], color='#0A345D', size=8, jitter=0.1)
axes[0].set_title('Acurácia por Fold (%)', fontweight='bold')
axes[0].set_ylabel('Acurácia (%)')

sns.boxplot(y=df_folds['Val_Macro_F1'], ax=axes[1], color='#7CB342', width=0.3)
sns.stripplot(y=df_folds['Val_Macro_F1'], ax=axes[1], color='#33691E', size=8, jitter=0.1)
axes[1].set_title('Macro F1 por Fold', fontweight='bold')
axes[1].set_ylabel('Macro F1')

plt.suptitle('Estabilidade e Variância entre os 5 Folds da ResNet-18', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 10. Diagnóstico Visual de Erros e Casos de Superconfiança

Inspecionamos os casos mais críticos do modelo: **erros cometidos com alta probabilidade predita ($P > 0.85$)**. Essa auditoria visual revela ambiguidades reais do dataset (como transições de pastagem para lavoura ou rios estreitos confundidos com rodovias).


In [ ]:
# Identifica amostras onde o modelo errou
is_error = (exp_labels != oof_predictions)
error_indices = np.where(is_error)[0]

# Probabilidade máxima predita nas amostras com erro
error_max_probs = np.max(oof_probabilities[error_indices], axis=1)

# Ordena pelos erros de maior confiança (superconfiança em predição errada)
worst_error_order = np.argsort(-error_max_probs)
top_worst_indices = error_indices[worst_error_order[:6]]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for plot_i, sample_idx in enumerate(top_worst_indices):
    img = Image.open(io.BytesIO(exp_images[sample_idx])).convert('RGB')
    real_label = CLASS_NAMES[exp_labels[sample_idx]]
    pred_label = CLASS_NAMES[oof_predictions[sample_idx]]
    confidence = oof_probabilities[sample_idx, oof_predictions[sample_idx]] * 100
    
    axes[plot_i].imshow(img)
    axes[plot_i].set_title(
        f"Real: {real_label}\nPredito: {pred_label} ({confidence:.1f}% conf.)",
        fontsize=10, fontweight='bold', color='#D32F2F'
    )
    axes[plot_i].axis('off')

plt.suptitle("🚨 Auditoria Visual: Top-6 Erros com Maior Grau de Superconfiança", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 11. Inferência em Produção com Ensemble de Folds (K-Fold Blending)

Ao invés de descartar 4 dos 5 modelos treinados na validação cruzada, a prática recomendada em produção é combinar os 5 modelos via **Média das Probabilidades Softmax (Ensemble Blending)**, garantindo maior estabilidade e redução de variância:
$$\mathbf{p}_{\text{ensemble}}(\mathbf{x}) = \frac{1}{K} \sum_{k=1}^K \mathbf{p}_{\text{model}_k}(\mathbf{x})$$


In [ ]:
# Carregamos os pesos dos 5 modelos treinados
ensemble_models = []
for fold in range(1, N_SPLITS + 1):
    m = create_model(num_classes=len(CLASS_NAMES), pretrained=False)
    m.load_state_dict(torch.load(f'checkpoints/best_model_fold_{fold}.pth', map_location=device))
    m.to(device)
    m.eval()
    ensemble_models.append(m)

def predict_single_image_ensemble(img_bytes, models_list, class_names):
    """
    Executa inferência em uma única imagem combinando as predições dos 5 folds.
    """
    img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    tensor = val_transforms(img).unsqueeze(0).to(device)
    
    all_probs = []
    with torch.no_grad():
        for m in models_list:
            out = m(tensor)
            prob = torch.softmax(out, dim=1).cpu().numpy()[0]
            all_probs.append(prob)
            
    avg_probs = np.mean(all_probs, axis=0)
    pred_class_id = np.argmax(avg_probs)
    return img, class_names[pred_class_id], avg_probs

# Demonstração em uma imagem de teste
test_idx = 42
test_img, pred_class, probs_dist = predict_single_image_ensemble(exp_images[test_idx], ensemble_models, CLASS_NAMES)
real_class = CLASS_NAMES[exp_labels[test_idx]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.imshow(test_img)
ax1.set_title(f"Imagem de Teste\nReal: {real_class}", fontsize=11, fontweight='bold')
ax1.axis('off')

# Gráfico de barras das probabilidades preditas pelo Ensemble
y_pos = np.arange(len(CLASS_NAMES))
colors_bar = ['#1BB5D8' if c != pred_class else '#7CB342' for c in CLASS_NAMES]
ax2.barh(y_pos, probs_dist * 100, color=colors_bar, edgecolor='#0A345D')
ax2.set_yticks(y_pos)
ax2.set_yticklabels(CLASS_NAMES, fontsize=9.5)
ax2.set_xlabel('Probabilidade Estimada pelo Ensemble (%)', fontsize=10)
ax2.set_title(f"Predição Final: {pred_class} ({probs_dist[np.argmax(probs_dist)]*100:.1f}%)", fontsize=11, fontweight='bold')
ax2.set_xlim(0, 100)

plt.suptitle('Demonstração de Inferência com Ensemble dos 5 Folds', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 12. Conclusão e Diretrizes Práticas para o Mundo Real

### 📋 Checklist de Melhores Práticas para Projetos de Visão Computacional:
1. **Adote Validação Cruzada Estratificada ($K$-Fold):** Evita o viés de divisão única, fornece intervalos de confiança de 95% e permite avaliar 100% dos dados via predições *Out-of-Fold*.
2. **Isole o Pipeline de Transforms:** O *Data Augmentation* pertence estritamente ao treino do fold corrente. Dados de validação nunca devem sofrer rotações/cortes artificiais.
3. **Não confie apenas na Acurácia Global:** Em cenários de desbalanceamento ou custos assimétricos, utilize o **Macro F1-Score**, analise a matriz de confusão sob as óticas de **Recall** e **Precisão**, e avalie as curvas **PR-AUC**.
4. **Audite os Erros Visuais:** Imagens com predições erradas e alta confiança indicam ruído de rotulagem (*label noise*) ou fronteiras ambíguas no domínio da aplicação.
5. **Utilize o Ensemble dos Folds em Produção:** Combinar as probabilidades dos $K$ checkpoints treinados na validação cruzada proporciona ganhos gratuitos de robustez e acurácia no deployment.
